# **UNIVERSIDADE FEDERAL DO CEARÁ**
---
Disciplina: Introdução à análise em Big Data

---

Professor: Luiz Alexandre

---

Alunos:
1.   Júlio César Gama Feitosa Freitas - 583956
2.   Vitória Freire Rocha Teixeira de Oliveira - 587661

---
Data: 13/09/2026

# 🧪 Lab 4 — Partições: organizar por ano/mês

## 🎯 Objetivo

Criar uma tabela particionada por `year`/`month` e medir a diferença de tempo entre uma busca com e sem partição.

**Importante:** **Simulação no Google Colab — Rota B (DuckDB + Parquet)**

## Configuração Inicial

In [5]:
# Instala o DuckDB
!pip -q install duckdb

In [6]:
# Faz o upload dos arquivos CSV necessários
# from google.colab import files
# uploaded = files.upload()
# print("Arquivos enviados:", list(uploaded.keys()))

In [7]:
# Configura o ambiente de arquivos e copia o CSV de transações
from pathlib import Path
import shutil

BASE = Path("../content/bigdata")
RAW = BASE / "raw" / "transactions"
SILVER = BASE / "silver"
RAW.mkdir(parents=True, exist_ok=True)
SILVER.mkdir(parents=True, exist_ok=True)

source = Path("../transactions_synthetic.csv")
if not source.exists():
    raise FileNotFoundError("Envie transactions_synthetic.csv e execute novamente esta célula.")

target = RAW / "transactions_synthetic.csv"
shutil.copy2(source, target)
print("CSV disponível em:", target)

CSV disponível em: ..\content\bigdata\raw\transactions\transactions_synthetic.csv


In [8]:
# Conecta-se ao DuckDB
import duckdb
import time

con = duckdb.connect("/content/bigdata_course.duckdb")
print("DuckDB conectado com sucesso.")

DuckDB conectado com sucesso.


## Passo 1 — Ler o CSV e criar colunas year/month

A coluna `timestamp` será convertida para `TIMESTAMP`. Depois extraímos o ano e o mês para serem usados como partições.

In [9]:
# Cria a VIEW 'raw_transactions' e extrai ano e mês
csv_path = str(target)

con.execute("DROP VIEW IF EXISTS raw_transactions")

sql = f"""
CREATE VIEW raw_transactions AS
SELECT *,
       EXTRACT(YEAR FROM CAST(timestamp AS TIMESTAMP))::INT AS year,
       EXTRACT(MONTH FROM CAST(timestamp AS TIMESTAMP))::INT AS month
FROM read_csv_auto('{csv_path}', header=true)
"""

con.execute(sql)
print("VIEW raw_transactions criada.")

VIEW raw_transactions criada.


In [10]:
# Verifica o total de transações na VIEW
total = con.execute("SELECT COUNT(*) FROM raw_transactions").fetchone()[0]
print("Total de transações:", total)
assert total == 100000, f"Esperado 100000, encontrado {total}"

Total de transações: 100000


In [11]:
# Lista as combinações únicas de ano e mês
partitions = con.execute("""
SELECT DISTINCT year, month
FROM raw_transactions
ORDER BY year, month
""").df()

print("Combinações ano/mês:", len(partitions))
display(partitions)

Combinações ano/mês: 24


,year,month
0,2023,1
1,2023,2
2,2023,3
3,2023,4
4,2023,5
5,2023,6
6,2023,7
7,2023,8
8,2023,9
9,2023,10


In [12]:
# Conta a quantidade de transações por mês e ano
monthly = con.execute("""
SELECT year, month, COUNT(*) AS quantidade
FROM raw_transactions
GROUP BY year, month
ORDER BY year, month
""").df()

display(monthly)

,year,month,quantidade
0,2023,1,4276
1,2023,2,3787
2,2023,3,4266
3,2023,4,4231
4,2023,5,4353
5,2023,6,4130
6,2023,7,4177
7,2023,8,4157
8,2023,9,4061
9,2023,10,4351


## Passo 2 — Gravar particionado de verdade em Parquet (DuckDB suporta isso nativamente)

O DuckDB criará uma estrutura como:

```text
transactions_particionado/
├── year=2023/
│   ├── month=1/
│   ├── month=2/
│   └── ...
└── year=2024/
    ├── month=1/
    └── ...
```

Isso é **particionamento físico real**, não apenas uma coluna calculada.

In [13]:
# Cria o diretório e copia as transações para Parquet particionado
partitioned_dir = SILVER / "transactions_particionado"

if partitioned_dir.exists():
    shutil.rmtree(partitioned_dir)

dest = str(partitioned_dir)
copy_sql = f"""
COPY raw_transactions
TO '{dest}'
(FORMAT PARQUET, PARTITION_BY (year, month))
"""
con.execute(copy_sql)

print("Parquet particionado criado em:", partitioned_dir)

Parquet particionado criado em: ..\content\bigdata\silver\transactions_particionado


## Passo 3 — Conferir a estrutura de pastas gerada

In [14]:
# Exibe a estrutura de pastas e arquivos Parquet criados
print("Estrutura criada:")
for p in sorted(partitioned_dir.rglob("*")):
    print(p.relative_to(partitioned_dir))

parquet_files = list(partitioned_dir.rglob("*.parquet"))
print("\nArquivos Parquet:", len(parquet_files))

Estrutura criada:
year=2023
year=2023\month=1
year=2023\month=1\data_0.parquet
year=2023\month=10
year=2023\month=10\data_0.parquet
year=2023\month=11
year=2023\month=11\data_0.parquet
year=2023\month=12
year=2023\month=12\data_0.parquet
year=2023\month=2
year=2023\month=2\data_0.parquet
year=2023\month=3
year=2023\month=3\data_0.parquet
year=2023\month=4
year=2023\month=4\data_0.parquet
year=2023\month=5
year=2023\month=5\data_0.parquet
year=2023\month=6
year=2023\month=6\data_0.parquet
year=2023\month=7
year=2023\month=7\data_0.parquet
year=2023\month=8
year=2023\month=8\data_0.parquet
year=2023\month=9
year=2023\month=9\data_0.parquet
year=2024
year=2024\month=1
year=2024\month=1\data_0.parquet
year=2024\month=10
year=2024\month=10\data_0.parquet
year=2024\month=11
year=2024\month=11\data_0.parquet
year=2024\month=12
year=2024\month=12\data_0.parquet
year=2024\month=2
year=2024\month=2\data_0.parquet
year=2024\month=3
year=2024\month=3\data_0.parquet
year=2024\month=4
year=2024\mont

## Passo 4 — Medir a diferença de tempo

**Sem partição:** consulta a `VIEW` sobre o CSV.

**Com partição:** aponta diretamente para `year=*/month=1/*.parquet`, lendo somente os arquivos de janeiro.

In [15]:
# Mede o tempo de busca por transações de janeiro sem usar partição
t0 = time.perf_counter()

result_without = con.execute("""
SELECT COUNT(*)
FROM raw_transactions
WHERE month = 1
""").fetchone()[0]

time_without = time.perf_counter() - t0

print("Janeiro — sem partição:", result_without)
print(f"Tempo sem partição: {time_without:.6f} s")

Janeiro — sem partição: 8461
Tempo sem partição: 0.105732 s


In [16]:
# Mede o tempo de busca por transações de janeiro usando partição
t0 = time.perf_counter()

parquet_glob = f"{partitioned_dir}/year=*/month=1/*.parquet"
result_with = con.execute(f"""
SELECT COUNT(*)
FROM read_parquet('{parquet_glob}')
""").fetchone()[0]

time_with = time.perf_counter() - t0

print("Janeiro — com partição:", result_with)
print(f"Tempo com partição: {time_with:.6f} s")

Janeiro — com partição: 8461
Tempo com partição: 0.017278 s


In [17]:
# Compara os resultados e tempos das buscas particionada e não-particionada
print("Os resultados são iguais?", result_without == result_with)
if time_with > 0:
    print(f"Razão de tempo (sem/com): {time_without / time_with:.2f}x")
assert result_without == result_with

Os resultados são iguais? True
Razão de tempo (sem/com): 6.12x


In [18]:
# Verifica a contagem de transações por ano/mês diretamente dos arquivos Parquet particionados
partition_check = con.execute(f"""
SELECT year, month, COUNT(*) AS quantidade
FROM read_parquet('{partitioned_dir}/year=*/month=*/*.parquet')
GROUP BY year, month
ORDER BY year, month
""").df()

display(partition_check)

,year,month,quantidade
0,2023,1,4276
1,2023,2,3787
2,2023,3,4266
3,2023,4,4231
4,2023,5,4353
5,2023,6,4130
6,2023,7,4177
7,2023,8,4157
8,2023,9,4061
9,2023,10,4351


## O que este Lab demonstra

Particionar significa organizar fisicamente os dados segundo atributos usados com frequência nos filtros.

```text
transactions → cria year/month → Parquet particionado
                                      ↓
                              year=2023/month=1/
                              year=2023/month=2/
                              ...
```

Quando uma consulta precisa apenas de janeiro, o mecanismo pode evitar a leitura das demais partições.

No Hive real, o conceito aparece em `PARTITIONED BY (year INT, month INT)`, `SHOW PARTITIONS` e `MSCK REPAIR TABLE`.

## ✅ Checkpoint

In [19]:
# Realiza verificações finais para confirmar a conclusão do Lab
print("=== CHECKPOINT LAB 4 ===")

checks = {
    "100.000 transações": total == 100000,
    "Partições encontradas": len(partitions) >= 1,
    "Parquet criado": partitioned_dir.exists() and len(parquet_files) > 0,
    "Consulta sem partição executada": result_without >= 0,
    "Consulta com partição executada": result_with >= 0,
    "Resultados iguais": result_without == result_with,
}

for item, ok in checks.items():
    print(("✅" if ok else "❌"), item)

assert all(checks.values())
print("\nLab 4 concluído com sucesso!")

=== CHECKPOINT LAB 4 ===
✅ 100.000 transações
✅ Partições encontradas
✅ Parquet criado
✅ Consulta sem partição executada
✅ Consulta com partição executada
✅ Resultados iguais

Lab 4 concluído com sucesso!
